<a href="https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/05_self_supervised_learning_and_transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Maritime Vision Applications
## Session 1 · Part 3 — Self-supervised pretraining & adapting foundation models
*Torben Globisch · VACOT*

---

| Notebook | Topic |
|---|---|
| 00 | Artificial Neural Networks: neurons, layers, learning |
| 01 | Computer Vision & an MLP on MNIST |
| 02 | Convolutional Neural Networks |
| 03 | Vision Foundation Models: the CNN era |
| 04 | Vision Transformers |
| **→ 05** | **Self-supervised pretraining & transfer** |

**By the end of this notebook you can**
1. explain how **MAE** and **DINO** pretrain a ViT **without any labels**,
2. use the dense features of **DINOv2** to segment boats with zero training,
3. **benchmark** frozen backbones with linear probing and k-NN,
4. choose between a linear head, a bigger head, partial and full **fine-tuning** (**Exercise**).

> 💡 Use a GPU runtime: *Runtime → Change runtime type → T4 GPU*. This notebook downloads ~600 MB of pretrained weights.

In [ ]:
# @title ⚙️ Setup: imports, colours & helpers — run me first { display-mode: "form" }
import math, io, time, urllib.request
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, FancyArrowPatch
from IPython.display import HTML, display
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

# Colour palette — the same colours as the lecture slides
NAVY, BLUE, LAVENDER, ORANGE, GREEN, RED, GREY, LIGHT = (
    "#2D3A8C", "#5B67B8", "#A3A9DB", "#E8702A", "#3E9E64", "#C0392B", "#8A8FA8", "#EEF0F7")
plt.rcParams.update({
    "figure.dpi": 100, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlecolor": NAVY,
    "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, GREEN, BLUE, RED, GREY]),
    "animation.html": "jshtml", "animation.embed_limit": 80,
})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)
print(f"PyTorch {torch.__version__} | device: {device}")
if device.type == "cpu":
    print("⚠️  No GPU found. In Colab: Runtime → Change runtime type → T4 GPU")

def play(anim):
    """Show a matplotlib animation as an HTML player (play / pause / step frame by frame)."""
    plt.close(anim._fig)
    return HTML(anim.to_jshtml())

# Maritime example photos from the COCO validation set (Flickr, Creative Commons licences).
MARITIME = {"trawler": 330369, "sailboat": 286422, "lifeboat": 400803, "marina": 442822,
            "fishing_boat": 451090, "yacht": 489014, "ferry": 543300, "harbour": 532575}

def load_image(name, size=None):
    """Load one of the MARITIME photos (or any image URL) as an RGB PIL image."""
    url = f"http://images.cocodataset.org/val2017/{MARITIME[name]:012d}.jpg" if name in MARITIME else name
    with urllib.request.urlopen(url, timeout=30) as r:
        img = Image.open(io.BytesIO(r.read())).convert("RGB")
    if size is not None:  # centre-crop to a square, then resize
        s = min(img.size); l, t = (img.width - s) // 2, (img.height - s) // 2
        img = img.crop((l, t, l + s, t + s)).resize((size, size), Image.BICUBIC)
    return img
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225])[:, None, None]

def to_tensor(img):
    """PIL → ImageNet-normalised tensor (3, H, W)."""
    return (torch.tensor(np.asarray(img), dtype=torch.float32).permute(2, 0, 1) / 255 - IMAGENET_MEAN) / IMAGENET_STD

def to_image(x):
    """ImageNet-normalised tensor (3, H, W) → displayable array (H, W, 3)."""
    return (x.cpu() * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

---
# 1 · Why self-supervised?
The ViT needs **a lot** of data (notebook 04). But labels are expensive: ImageNet took years of crowd-sourcing, and in the
maritime domain annotations often need experts (vessel types, sonar imagery, underwater species, …).
Unlabelled images, on the other hand, are cheap: every camera on every ship produces them all day.

**Self-supervised learning (SSL)** invents a *pretext task* whose "labels" come from the data itself:

| Method | Pretext task | Family |
|---|---|---|
| **MAE** (He et al., 2021, Meta) | hide 75 % of the patches, reconstruct the pixels | masked image modelling |
| **DINO** (Caron et al., 2021, Meta) | two random crops of the same image must give the same output | self-distillation |
| **DINOv2** (Oquab et al., 2023, Meta) | DINO + masked modelling, at scale (142 M curated images) | both |

---
# 2 · Masked Autoencoders (MAE)
**How it works**
1. divide the image into patches (as in every ViT),
2. **mask 75 %** of the patches at random,
3. the **encoder** processes only the visible 25 % (cheap!),
4. a light **decoder** reconstructs the pixel values of the masked patches,
5. **loss:** MSE on the masked patches only.

**Why 75 %?** Images are highly redundant: with only a few patches missing, the model could just interpolate from neighbours.
Hiding most of the image forces it to *understand the scene*.

In [ ]:
# @title 🎞️ Animation: how much of the image can we hide? { display-mode: "form" }
img = load_image("trawler", size=224)
arr = np.asarray(img) / 255
order = np.random.default_rng(0).permutation(196)          # random masking order of the 14×14 patches
fig, ax = plt.subplots(figsize=(5, 5.3)); ax.axis("off"); im = ax.imshow(arr); ttl = ax.set_title("")
ratios = list(np.arange(0, 0.91, 0.05)) + [0.9] * 4 + [0.75] * 6
def update(f):
    r = ratios[f]; masked = arr.copy()
    for k in order[:int(round(r * 196))]:
        i, j = divmod(k, 14); masked[i * 16:(i + 1) * 16, j * 16:(j + 1) * 16] = 0.5
    im.set_data(masked); ttl.set_text(f"mask ratio {r:.0%}" + ("  ← MAE default" if abs(r - .75) < 1e-6 and f >= len(ratios) - 6 else ""))
    return [im]
play(animation.FuncAnimation(fig, update, frames=len(ratios), interval=300))

Even at 75 % *you* can still guess that this is a boat in a harbour. That kind of guessing is exactly the skill MAE trains.
Let's ask a real pretrained MAE (ViT-Base) to fill in the gaps:

In [ ]:
from transformers import ViTMAEForPreTraining
mae = ViTMAEForPreTraining.from_pretrained("facebook/vit-mae-base").eval().to(device)

MASK_RATIO = 0.75          # ✏️ try 0.5 or 0.9
mae.config.mask_ratio = MASK_RATIO
mae.vit.embeddings.config.mask_ratio = MASK_RATIO

names = ["trawler", "sailboat", "lifeboat", "fishing_boat"]
x = torch.stack([to_tensor(load_image(n, size=224)) for n in names]).to(device)
torch.manual_seed(0)
with torch.no_grad():
    out = mae(pixel_values=x)
recon = mae.unpatchify(out.logits)                                                 # (B, 3, 224, 224) predicted pixels
mask = out.mask[:, :, None].repeat(1, 1, 16 * 16 * 3)                              # (B, 196, 768) 1 = masked
mask = mae.unpatchify(mask)                                                        # (B, 3, 224, 224)
masked_input = x * (1 - mask) + mask * ((0.5 - IMAGENET_MEAN.to(device)) / IMAGENET_STD.to(device))   # grey for hidden patches
pasted = x * (1 - mask) + recon * mask                                             # visible patches + reconstructed ones

fig, axes = plt.subplots(len(names), 4, figsize=(12, 12))
for r in range(len(names)):
    for c, (t, title) in enumerate([(x, "original"), (masked_input, f"input to the encoder ({MASK_RATIO:.0%} masked)"),
                                    (recon, "reconstruction"), (pasted, "reconstruction + visible patches")]):
        axes[r, c].imshow(to_image(t[r])); axes[r, c].axis("off")
        if r == 0: axes[r, c].set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

The reconstructions are blurry (MSE loss averages over all plausible answers), but the model clearly "knows" where the hull,
the water and the horizon go, *from a quarter of the patches and without ever seeing a label*.
**Result:** MAE features are excellent when **fine-tuned**, e.g. for global prediction tasks.

---
# 3 · DINO: self-distillation with no labels
**Key idea:** make the network **agree with itself** on different views of the same image.

1. generate several random **crops** of the same image: 2 large *global* views + several small *local* views,
2. the **student** (a ViT, trained by gradient descent) sees **all** crops,
3. the **teacher** (same architecture) sees only the global views; its weights are an **exponential moving average (EMA)**
   of the student's, so it gets no gradient at all,
4. **loss:** cross-entropy $H(p_t, p_s)$ between the teacher's and the student's output distributions,
5. no labels required! (Centering and sharpening of the teacher outputs prevent the trivial solution "always output the same".)

In [ ]:
from torchvision import transforms
global_crop = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.4, 1.0)), transforms.RandomHorizontalFlip(),
                                  transforms.ColorJitter(0.4, 0.4, 0.2, 0.1)])
local_crop = transforms.Compose([transforms.RandomResizedCrop(96, scale=(0.05, 0.4)), transforms.RandomHorizontalFlip(),
                                 transforms.ColorJitter(0.4, 0.4, 0.2, 0.1)])
base = load_image("trawler")
torch.manual_seed(1)
fig, axes = plt.subplots(1, 7, figsize=(16, 3), gridspec_kw=dict(width_ratios=[1.5, 1.2, 1.2, .7, .7, .7, .7]))
axes[0].imshow(base); axes[0].set_title("original")
for k in range(2): axes[1 + k].imshow(global_crop(base)); axes[1 + k].set_title(f"global view {k + 1}\n→ teacher & student", color=NAVY, fontsize=9)
for k in range(4): axes[3 + k].imshow(local_crop(base)); axes[3 + k].set_title(f"local {k + 1}\n→ student", color=ORANGE, fontsize=9)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

The task for the student: *"from this small local crop, predict what the teacher says about the whole image"*.
To do that it has to learn what the object **is**, not what the pixels look like.

The teacher update is a single line, no backpropagation involved:

In [ ]:
@torch.no_grad()
def ema_update(teacher, student, momentum=0.996):
    for p_t, p_s in zip(teacher.parameters(), student.parameters()):
        p_t.mul_(momentum).add_((1 - momentum) * p_s)        # θ_teacher ← m·θ_teacher + (1 − m)·θ_student

### What does DINO learn? Look at the [CLS] attention
Below: the last-layer [CLS] attention of each of the 6 heads of a DINO ViT-S/8. Remember the noisy maps of the
supervised ViT in notebook 04? DINO was **never told what a boat is**, yet its heads segment the objects.

In [ ]:
from transformers import ViTModel
dino = ViTModel.from_pretrained("facebook/dino-vits8", attn_implementation="eager", add_pooling_layer=False).eval().to(device)

SIZE = 320                                                        # 320 / 8 = 40 × 40 patches
dino_names = ["trawler", "sailboat", "lifeboat", "ferry"]
xs = torch.stack([to_tensor(load_image(n, size=SIZE)) for n in dino_names]).to(device)
with torch.no_grad():
    att = dino(pixel_values=xs, output_attentions=True, interpolate_pos_encoding=True).attentions[-1]   # (B, 6, 1+1600, 1+1600)
cls_att = att[:, :, 0, 1:].reshape(len(dino_names), 6, SIZE // 8, SIZE // 8).cpu()

fig, axes = plt.subplots(len(dino_names), 7, figsize=(16, 9.6))
for r in range(len(dino_names)):
    axes[r, 0].imshow(to_image(xs[r]))
    for h in range(6):
        a = cls_att[r, h]; a = a / a.max()
        axes[r, h + 1].imshow(a, cmap="inferno")
        if r == 0: axes[r, h + 1].set_title(f"head {h + 1}")
for ax in axes.ravel(): ax.axis("off")
plt.suptitle("DINO ViT-S/8: last-layer [CLS] attention per head (no labels used in training!)", color=NAVY, fontweight="bold")
plt.tight_layout(); plt.show()

---
# 4 · DINOv2: dense features you can use directly
**DINOv2** (2023) scaled the recipe up (142 M curated images, ViT up to 1 B parameters) and added a masked-modelling loss.
Its **patch tokens** are so good that simple methods on top of *frozen* features work for retrieval, depth estimation and segmentation.

A popular way to *see* dense features: run **PCA** on all patch tokens and show the first three components as RGB.
Patches that the model considers similar get similar colours:

In [ ]:
from transformers import AutoModel
dinov2 = AutoModel.from_pretrained("facebook/dinov2-small").eval().to(device)

SIZE2, P2 = 448, 14                                              # 448 / 14 = 32 × 32 patch tokens
pca_names = ["trawler", "sailboat", "marina", "harbour", "fishing_boat", "ferry"]
x2 = torch.stack([to_tensor(load_image(n, size=SIZE2)) for n in pca_names]).to(device)
with torch.no_grad():
    tokens = dinov2(pixel_values=x2).last_hidden_state[:, 1:].cpu()   # (B, 1024, 384): drop [CLS]
G = SIZE2 // P2
feats = tokens.reshape(-1, tokens.shape[-1])                      # all patches of all images

# 1st principal component → foreground vs. background
from sklearn.decomposition import PCA
pc1 = PCA(n_components=1).fit_transform(feats.numpy())[:, 0]
if np.median(pc1) > 0: pc1 = -pc1                                  # make the (larger) background negative
fg = pc1 > 0
# PCA with 3 components on the foreground patches only → RGB
pca3 = PCA(n_components=3).fit(feats[fg].numpy())
rgb = np.zeros((len(feats), 3)); rgb[fg] = pca3.transform(feats[fg].numpy())
rgb[fg] = (rgb[fg] - rgb[fg].min(0)) / (rgb[fg].max(0) - rgb[fg].min(0))

fig, axes = plt.subplots(3, len(pca_names), figsize=(16, 8))
for k in range(len(pca_names)):
    s = slice(k * G * G, (k + 1) * G * G)
    axes[0, k].imshow(to_image(x2[k]))
    axes[1, k].imshow(fg[s].reshape(G, G), cmap="gray")
    axes[2, k].imshow(rgb[s].reshape(G, G, 3))
for ax in axes.ravel(): ax.axis("off")
axes[0, 0].set_title("image", loc="left"); axes[1, 0].set_title("1st PC > 0: 'foreground'", loc="left"); axes[2, 0].set_title("PCA of foreground patches → RGB", loc="left")
plt.suptitle("DINOv2 patch features: segmentation with zero labels and zero training", color=NAVY, fontweight="bold")
plt.tight_layout(); plt.show()

With **no labels and no training at all**, a single PCA separates boats from water and sky. The same object parts get
the same colour across *different* images (hulls, masts, cabins, …), so the features are **semantic** and consistent between images.

---
# 5 · Benchmarking: evaluating frozen representations
**How do we know whether pretraining was good?**

The gold standard is to keep the backbone **frozen** (no gradient flows into it) and train only a tiny model on top:
* **linear probe:** a single linear layer on the features,
* **k-NN:** classify each test image by a majority vote of its nearest training images in feature space (no training at all).

*Why frozen?* Fine-tuning can compensate for bad representations. Frozen evaluation isolates the quality of the pretrained
features themselves. Common benchmarks: ImageNet top-1 (linear), COCO detection, ADE20k segmentation (linear).

We benchmark three frozen backbones on the same small CIFAR-10 setup as notebook 03 (5,000 labelled training images):

In [ ]:
import torchvision
from torchvision import datasets
from torch.utils.data import TensorDataset, DataLoader
cifar_train = datasets.CIFAR10("data", train=True, download=True)
cifar_test = datasets.CIFAR10("data", train=False, download=True)
CIFAR_CLASSES = cifar_train.classes
N_TRAIN, N_TEST = 5000, 1000
g = torch.Generator().manual_seed(0)
idx_train = torch.randperm(len(cifar_train), generator=g)[:N_TRAIN]; idx_test = torch.randperm(len(cifar_test), generator=g)[:N_TEST]
imgs_train = torch.tensor(cifar_train.data[idx_train]).permute(0, 3, 1, 2); labels_train = torch.tensor(cifar_train.targets)[idx_train]
imgs_test = torch.tensor(cifar_test.data[idx_test]).permute(0, 3, 1, 2); labels_test = torch.tensor(cifar_test.targets)[idx_test]

def prep(x_uint8, size=224):
    x = F.interpolate(x_uint8.to(device).float() / 255, size=size, mode="bilinear", align_corners=False)
    return (x - IMAGENET_MEAN.to(device)) / IMAGENET_STD.to(device)

@torch.no_grad()
def extract(fn, imgs, batch_size=200):
    return torch.cat([fn(prep(imgs[i:i + batch_size])).float().cpu() for i in range(0, len(imgs), batch_size)])

resnet_sup = torchvision.models.resnet50(weights="IMAGENET1K_V2"); resnet_sup.fc = nn.Identity(); resnet_sup.eval().to(device)
resnet_rand = torchvision.models.resnet50(weights=None); resnet_rand.fc = nn.Identity(); resnet_rand.eval().to(device)

def dinov2_features(x):
    h = dinov2(pixel_values=x).last_hidden_state
    return torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=1)        # [CLS] ‖ mean of patch tokens (the DINOv2 linear-eval recipe)

backbones = {
    "ResNet-50, random init\n(no pretraining)": (resnet_rand, 25.6),
    "ResNet-50, supervised\n(ImageNet labels)": (resnet_sup, 25.6),
    "DINOv2 ViT-S/14\n(self-supervised)": (dinov2_features, 22.1),
}
features = {}
for name, (fn, _) in backbones.items():
    t0 = time.time(); features[name] = (extract(fn, imgs_train), extract(fn, imgs_test))
    print(f"{name.replace(chr(10), ' '):<55} dim {features[name][0].shape[1]}  ({time.time() - t0:.0f}s)")

### ✏️ Implement the k-NN classifier
For each test feature: cosine similarity to all training features → take the `k` most similar → majority vote of their labels.

In [ ]:
def knn_accuracy(f_train, y_train, f_test, y_test, k=20):
    # TODO: 1) L2-normalise both feature sets (F.normalize)  2) similarity matrix  3) topk  4) majority vote (torch.mode)
    return float("nan")

In [ ]:
# @title 💡 Solution { display-mode: "form" }
def knn_accuracy(f_train, y_train, f_test, y_test, k=20):
    sim = F.normalize(f_test, dim=1) @ F.normalize(f_train, dim=1).T     # (n_test, n_train) cosine similarities
    nn_labels = y_train[sim.topk(k, dim=1).indices]                      # (n_test, k)
    pred = nn_labels.mode(dim=1).values                                  # majority vote
    return (pred == y_test).float().mean().item()

In [ ]:
def linear_probe(f_train, y_train, f_test, y_test, epochs=100):
    mu, sd = f_train.mean(0), f_train.std(0) + 1e-6                      # standardise features
    f_train, f_test = (f_train - mu) / sd, (f_test - mu) / sd
    probe = nn.Linear(f_train.shape[1], 10); opt = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=1e-3)
    for _ in range(epochs):
        for f, y in DataLoader(TensorDataset(f_train, y_train), batch_size=256, shuffle=True):
            loss = F.cross_entropy(probe(f), y); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): return (probe(f_test).argmax(1) == y_test).float().mean().item()

torch.manual_seed(0)
results = {name: (knn_accuracy(ftr, labels_train, fte, labels_test), linear_probe(ftr, labels_train, fte, labels_test))
           for name, (ftr, fte) in features.items()}
for name, (knn, lin) in results.items(): print(f"{name.replace(chr(10), ' '):<55} k-NN {knn:.1%}   linear {lin:.1%}")

fig, ax = plt.subplots(figsize=(10, 4))
xpos = np.arange(len(results))
for k, (lbl, col) in enumerate([("k-NN (k = 20)", LAVENDER), ("linear probe", NAVY)]):
    vals = [v[k] for v in results.values()]
    b = ax.bar(xpos + (k - .5) * .36, vals, .34, color=col, label=lbl); ax.bar_label(b, labels=[f"{v:.0%}" for v in vals], fontsize=9)
ax.set_xticks(xpos, [f"{n}\n{backbones[n][1]} M params" for n in results]); ax.set_ylim(0, 1.05)
ax.set_ylabel("CIFAR-10 test accuracy"); ax.legend(loc="upper left"); ax.set_title("Frozen backbones, 5,000 labels: how good are the features?")
plt.tight_layout(); plt.show()

* A **random** network gives poor features, so *the architecture alone* is not the magic. Pretraining is.
* The **self-supervised** DINOv2 ViT-S (never saw a single label during pretraining, and slightly *smaller* than ResNet-50)
  gives the best features. Even the training-free k-NN classifier gets most images right.

---
# 6 · Applying VFMs: from linear heads to fine-tuning

| | trained parameters | when to use |
|---|---|---|
| **linear head** on frozen features | tiny ($d \times C$) | features already encode what you need; very few labels |
| **bigger head** (MLP, FPN, UPerNet, …) on frozen features | small | dense prediction, multi-scale fusion, non-linear tasks |
| **partial fine-tuning** (unfreeze the last $k$ blocks) | medium | domain shift, e.g. ImageNet → maritime/sonar/satellite |
| **full fine-tuning** | everything | lots of labels, large domain shift; needs a small learning rate |

*less adaptation ← linear · bigger head · partial · full fine-tuning → more adaptation*

**Dense prediction with a linear head:** keep the patch tokens instead of [CLS], reshape them to the spatial grid
($32\times32\times d$ above) and apply a **1 × 1 convolution**, which is the same linear layer applied to every patch:

In [ ]:
d, n_classes = 384, 5
patch_grid = tokens[:1].reshape(1, G, G, d).permute(0, 3, 1, 2)          # (1, 384, 32, 32)
conv1x1 = nn.Conv2d(d, n_classes, kernel_size=1)
linear = nn.Linear(d, n_classes); linear.weight.data = conv1x1.weight.data[:, :, 0, 0]; linear.bias.data = conv1x1.bias.data
same = torch.allclose(conv1x1(patch_grid), linear(patch_grid.permute(0, 2, 3, 1)).permute(0, 3, 1, 2), atol=1e-5)
print("per-pixel class map:", tuple(conv1x1(patch_grid).shape), "| 1×1 conv == per-patch linear layer:", same)

### ✏️ Exercise: linear probe vs. fine-tuning with very few labels
Can we do better than the frozen linear probe by also training (part of) the backbone? We take only **1,000 labelled images**
(100 per class) and compare four strategies on DINOv2 ViT-S. *(≈ 3 min on a T4 GPU.)*

Your job: implement `set_trainable`, which freezes the whole backbone and then unfreezes only the last `k` Transformer blocks
(`k = 0`: frozen backbone; `k = 12`: full fine-tuning). Hints: `p.requires_grad_(False)`, the blocks live in `backbone.encoder.layer`.

In [ ]:
def set_trainable(backbone, k):
    # TODO 1: freeze ALL backbone parameters
    # TODO 2: unfreeze the parameters of the last k blocks in backbone.encoder.layer (and backbone.layernorm if k > 0)
    pass

In [ ]:
# @title 💡 Solution { display-mode: "form" }
def set_trainable(backbone, k):
    for p in backbone.parameters():
        p.requires_grad_(False)
    if k > 0:
        for block in backbone.encoder.layer[-k:]:
            for p in block.parameters(): p.requires_grad_(True)
        for p in backbone.layernorm.parameters(): p.requires_grad_(True)

In [ ]:
class Classifier(nn.Module):
    def __init__(self, backbone, head="linear", n_classes=10):
        super().__init__()
        self.backbone = backbone; d = backbone.config.hidden_size
        self.head = nn.Linear(2 * d, n_classes) if head == "linear" else nn.Sequential(
            nn.Linear(2 * d, 512), nn.GELU(), nn.Dropout(0.2), nn.Linear(512, n_classes))
    def forward(self, x):
        h = self.backbone(pixel_values=x).last_hidden_state
        return self.head(torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=1))

def run_strategy(head, k_blocks, n_per_class=100, epochs=6, lr_head=1e-3, lr_backbone=2e-5, size=224):
    torch.manual_seed(0)
    idx = torch.cat([(labels_train == c).nonzero()[:n_per_class, 0] for c in range(10)])
    model = Classifier(AutoModel.from_pretrained("facebook/dinov2-small"), head).to(device)
    set_trainable(model.backbone, k_blocks)
    params = [{"params": model.head.parameters(), "lr": lr_head}]
    bb = [p for p in model.backbone.parameters() if p.requires_grad]
    if bb: params.append({"params": bb, "lr": lr_backbone})
    opt = torch.optim.AdamW(params, weight_decay=0.05)
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    loader = DataLoader(TensorDataset(imgs_train[idx], labels_train[idx]), batch_size=32, shuffle=True)
    flip = transforms.RandomHorizontalFlip(); t0 = time.time()
    for epoch in range(epochs):
        model.train(); model.backbone.train(k_blocks > 0)
        for xb, yb in loader:
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                loss = F.cross_entropy(model(prep(flip(xb), size)), yb.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); correct = 0
    with torch.no_grad(), torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
        for i in range(0, N_TEST, 200):
            correct += (model(prep(imgs_test[i:i + 200], size)).argmax(1).cpu() == labels_test[i:i + 200]).sum().item()
    return correct / N_TEST, n_trainable, time.time() - t0

strategies = {"linear head\n(frozen)": ("linear", 0), "MLP head\n(frozen)": ("mlp", 0),
              "fine-tune last 2\nblocks + head": ("linear", 2), "full fine-tuning": ("linear", 12)}
ft_results = {}
for name, (head, k) in strategies.items():
    ft_results[name] = run_strategy(head, k)
    acc, n, secs = ft_results[name]
    print(f"{name.replace(chr(10), ' '):<34} acc {acc:.1%}   trainable params {n:>10,}   ({secs:.0f}s)")

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
names_ft = list(ft_results); accs = [ft_results[n][0] for n in names_ft]; ntr = [ft_results[n][1] for n in names_ft]
b = ax1.bar(names_ft, accs, color=[LAVENDER, BLUE, ORANGE, RED], width=.55); ax1.bar_label(b, labels=[f"{a:.1%}" for a in accs], fontweight="bold")
ax1.set_ylim(min(accs) - .08, 1.0); ax1.set_ylabel("CIFAR-10 test accuracy (1,000 labels)")
ax2 = ax1.twinx(); ax2.plot(names_ft, ntr, "o--", color=NAVY); ax2.set_yscale("log"); ax2.set_ylabel("trainable parameters (log)", color=NAVY)
ax2.spines["right"].set_visible(True)
ax1.set_title("Less ← adaptation → more"); plt.tight_layout(); plt.show()

Compare the bars with the parameter counts. With only 1,000 labels and an in-domain-ish task, a head on frozen features
is very hard to beat, and it trains tens of thousands of parameters instead of 22 million. Adapting the backbone pays off
when the **domain shift** is large (sonar, infrared, underwater imagery, aerial views of the sea, …) and enough labels
are available. That is a question you will meet again in your own maritime projects.

### 🔬 Experiments
1. Increase `n_per_class` to 500. Does fine-tuning catch up?
2. Set `lr_backbone=1e-3` for full fine-tuning. What happens to the pretrained knowledge? (*catastrophic forgetting*)
3. Swap `facebook/dinov2-small` for `facebook/dinov2-base` (86 M parameters).

---
# 7 · Closing: "how foundational is it?"
The term *foundation model* has no single agreed definition. Models differ not in **kind** but in **degree**. Ask these
questions about any model you meet:

| | ResNet-50 (ImageNet, 2015) | MAE ViT-B (2021) | DINOv2 ViT-S (2023) |
|---|---|---|---|
| pretraining data | 1.28 M images, **labelled** | 1.28 M images, **no labels** | 142 M curated images, **no labels** |
| how many downstream tasks? | classification; detection/segmentation after fine-tuning | strong after fine-tuning | classification, retrieval, segmentation, depth, … with frozen features |
| adaptation needed | fine-tuning per task | fine-tuning | often just a linear head |
| transfer to unseen domains | moderate | good | very good (still: test it on *your* data!) |

**For maritime vision we are at the frontier.** The core concepts from today (pretraining, frozen evaluation, domain
adaptation) are the tools we will apply throughout this course.

---
# Summary
1. **Self-supervised learning** creates labels from the data itself and makes pretraining on unlabelled images possible.
2. **MAE:** mask 75 % of the patches, reconstruct the pixels; the encoder only sees the visible ones.
3. **DINO:** a student matches an EMA teacher across different crops; its attention segments objects without labels.
4. **DINOv2** patch features are semantic enough to segment boats with a single PCA.
5. **Frozen evaluation** (linear probe, k-NN) measures representation quality; self-supervised features can beat supervised ones.
6. **Adaptation spectrum:** linear head → bigger head → partial → full fine-tuning; choose by data size and domain shift.

> **Take-home message:** the field has moved from *training from scratch* to *adapting pretrained representations*.
> Understanding what a model learned during pretraining is as important as understanding its architecture.

**Next session:** Scientific writing for ML papers: how to structure a methods section, and how to cite and ablate VFMs correctly.